# Flash Evaluation on the LANL 2015 Dataset
This notebook is designed for evaluating Flash on the LANL 2015 dataset. Coded by zyx.

## Data Parsing and Execution:
- The script is adept at autonomously parsing the downloaded data files.
- For evaluation results, execute all cells in this notebook.

## Model Training and Execution Options:
- By default, the notebook utilizes pre-trained model weights.
- It also offer settings to independently train Graph Neural Networks (GNNs), word2vec, and Xgboost models.
- These independently trained models can then be deployed for an evaluation of the system.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import torch
from torch_geometric.data import Data
import os
import torch.nn.functional as F
import pickle
import json
import warnings
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
warnings.filterwarnings('ignore')
from torch_geometric.loader import NeighborLoader
from collections import defaultdict
from gensim.models import Word2Vec
import xgboost as xgb
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
%matplotlib inline

In [6]:
gnn_weights = "trained_weights/optc/gnn_temp.pth"
xgboost_weights = "trained_weights/optc/xgb.pkl"
word2vec_weights = 'w2v_optc.model'
# 训练控制标志
gnnTrain = True
create_store = True
xgbTrain = True

In [7]:
# LANL 原始数据路径
AUTH_FILE = "./datasets/lanl/auth.txt"   # 请修改为实际路径
RED_TEAM_FILE = "./datasets/lanl/redteam.txt"

# 划分时间点（与 lanl_parser.py 一致）
DATE_OF_EVIL_LANL = 150885   # 训练集截止时间（秒）
END_TEST_TIME = 1270000      # 测试集结束时间（秒）

# 模型保存路径
gnn_weights = "trained_weights/lanl/gnn.pth"
xgboost_weights = "trained_weights/lanl/xgb.pkl"
word2vec_weights = "trained_weights/lanl/w2v_lanl.model"
emb_store_path = "data_files/lanl_emb_store.json"

In [8]:
def build_phrase_from_event(event):
    """
    从一条认证日志构造短语（词列表）
    event: 字典，包含字段：
        'auth_type', 'logon_type', 'orientation', 'result'
    返回字符串列表，例如 ['Negotiate', 'Service', 'LogOn', 'Success']
    """
    words = []
    if 'auth_type' in event and event['auth_type']:
        words.append(event['auth_type'])
    if 'logon_type' in event and event['logon_type']:
        words.append(event['logon_type'])
    if 'orientation' in event and event['orientation']:
        words.append(event['orientation'])
    if 'result' in event and event['result']:
        words.append(event['result'])
    return words


# print("Training Word2Vec on ALL data (train + test)")
# all_phrases = []
# with open(AUTH_FILE, 'r') as f:
#     for line in f:
#         parts = line.strip().split(',')
#         if len(parts) < 9 or 'NTLM' not in line.upper():
#             continue
#         ts = int(parts[0])
#         if ts >= END_TEST_TIME:   # 只使用我们划分的时间范围内的数据
#             continue
#         # 解析字段并构造短语
#         phrase = build_phrase_from_event({
#             'auth_type': parts[5], 'logon_type': parts[6],
#             'orientation': parts[7], 'result': parts[8]
#         })
#         if phrase:
#             all_phrases.append(phrase)

# # 训练 Word2Vec 并保存
# tmp_model = Word2Vec(sentences=all_phrases, vector_size=20, window=5, min_count=1, workers=8, epochs=100)
# tmp_model.save(word2vec_weights)
# print("Word2Vec trained on all data and saved.")
# # 不要赋值给 w2vmodel

In [9]:
import math
import torch
import numpy as np

class PositionalEncoder:

    def __init__(self, d_model, max_len=100000):
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)

    def embed(self, x):
        return x + self.pe[:x.size(0)]

# encoder = PositionalEncoder(20)
# w2vmodel = Word2Vec.load(word2vec_weights)

In [10]:
# 全局 Word2Vec 模型和编码器（将在训练后加载）
w2vmodel = None
encoder = PositionalEncoder(20)

def get_malicious_in_range(start, end):
    mal = set()
    with open(RED_TEAM_FILE, 'r') as rf:
        for rline in rf:
            rparts = rline.strip().split(',')
            if len(rparts) >= 4:
                t = int(rparts[0])
                if start <= t < end:
                    mal.add(rparts[2])
                    mal.add(rparts[3])
    return mal

def infer(document):
    """将文档（词列表列表）转换为 20 维特征（平均词向量+位置编码）"""
    word_vecs = []
    for phrase in document:
        for word in phrase:
            if word in w2vmodel.wv:
                word_vecs.append(w2vmodel.wv[word])
    if not word_vecs:
        return np.zeros(20)
    emb = torch.tensor(word_vecs, dtype=torch.float)
    if len(word_vecs) < 100000:
        emb = encoder.embed(emb)
    return np.mean(emb.numpy(), axis=0)


_cache = {}

def load_lanl_data(split='train', label_type='degree'):
    """
    加载 LANL 数据，严格按时间划分。
    - split: 'train' or 'test'
    - label_type: 
        'degree'   -> 返回基于节点度数的伪标签（用于 GNN 训练）
        'malicious' -> 返回二分类恶意标签（用于 XGBoost 训练/测试）
    返回: features, labels, edge_index, mapping, lbl_map, nemap
    """
    if (split, label_type) in _cache:
        return _cache[(split, label_type)]
    
    global w2vmodel
    if w2vmodel is None:
        w2vmodel = Word2Vec.load(word2vec_weights)   # 加载预先训练好的模型

    node_phrases = defaultdict(list)   # 用于生成节点特征，只使用训练时间短语
    events = []                        # 用于构建边和 nemap，根据 split 筛选

    with open(AUTH_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 9 or 'NTLM' not in line.upper():
                continue
            ts = int(parts[0])
            src_com = parts[3]
            dst_com = parts[4]
            # 构造短语
            phrase = build_phrase_from_event({
                'auth_type': parts[5], 'logon_type': parts[6],
                'orientation': parts[7], 'result': parts[8]
            })
            if not phrase:
                continue

            # node_phrases 始终只使用训练时间事件 (ts < DATE_OF_EVIL_LANL)
            if ts < DATE_OF_EVIL_LANL:
                node_phrases[src_com].append(phrase)
                node_phrases[dst_com].append(phrase)

            # events 根据 split 严格筛选时间
            if split == 'train' and ts < DATE_OF_EVIL_LANL:
                event = {
                    'actorID': src_com,
                    'objectID': dst_com,
                    'object': 'AUTH',
                    'action': 'Logon',
                    'timestamp': ts,
                    'properties': {
                        'auth_type': parts[5], 'logon_type': parts[6],
                        'orientation': parts[7], 'result': parts[8],
                        'src_user': parts[1], 'dst_user': parts[2]
                    }
                }
                events.append(event)
            elif split == 'test' and DATE_OF_EVIL_LANL <= ts < END_TEST_TIME:
                event = {
                    'actorID': src_com,
                    'objectID': dst_com,
                    'object': 'AUTH',
                    'action': 'Logon',
                    'timestamp': ts,
                    'properties': {
                        'auth_type': parts[5], 'logon_type': parts[6],
                        'orientation': parts[7], 'result': parts[8],
                        'src_user': parts[1], 'dst_user': parts[2]
                    }
                }
                events.append(event)

    # 构建节点特征
    node_list = list(node_phrases.keys())
    node_index = {node: idx for idx, node in enumerate(node_list)}
    features = []
    for node in node_list:
        feat = infer(node_phrases[node])
        features.append(feat)

    # 构建边
    edge_index = [[], []]
    for ev in events:
        src, dst = ev['actorID'], ev['objectID']
        if src in node_index and dst in node_index:
            edge_index[0].append(node_index[src])
            edge_index[1].append(node_index[dst])

    # ========== 根据 label_type 生成不同的标签 ==========
    if label_type == 'malicious':
        # 二分类恶意标签（用于 XGBoost）
        if split == 'train':
            mal_nodes = get_malicious_in_range(0, DATE_OF_EVIL_LANL)
        else:
            mal_nodes = get_malicious_in_range(DATE_OF_EVIL_LANL, END_TEST_TIME)
        labels = np.array([1 if node in mal_nodes else 0 for node in node_list])
    else:   # 'degree' 伪标签（用于 GNN 训练）
        # 计算每个节点的度数（基于 events 中的边）
        degree = {node: 0 for node in node_list}
        for ev in events:
            src, dst = ev['actorID'], ev['objectID']
            if src in node_index:
                degree[src] += 1
            if dst in node_index:
                degree[dst] += 1
        deg_vals = np.array([degree[node] for node in node_list])
        # 按三分位数划分伪标签（可调整类别数量，此处设为3类）
        q33, q66 = np.percentile(deg_vals, [33, 66])
        labels = np.zeros(len(node_list), dtype=int)
        labels[deg_vals > q66] = 2
        labels[(deg_vals > q33) & (deg_vals <= q66)] = 1
        # labels[deg_vals <= q33] = 0 already
        print(f"Degree pseudo-label distribution: {np.bincount(labels)}")

    # 构建 nemap（只基于 events 中的边）
    nemap = {node: set() for node in node_list}
    for ev in events:
        src, dst = ev['actorID'], ev['objectID']
        if src in node_index and dst in node_index:
            nemap[src].add(dst)
            nemap[dst].add(src)

    mapping = node_list
    lbl_map = {node: node for node in node_list}

    _cache[(split, label_type)] = (features, labels, edge_index, mapping, lbl_map, nemap)
    return _cache[(split, label_type)]

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GCN(torch.nn.Module):
    def __init__(self, in_dim=20, hidden=32, num_classes=3):   # 增加 num_classes 参数
        super(GCN, self).__init__()
        self.conv1 = SAGEConv(in_dim, hidden, normalize=True)
        self.conv2 = SAGEConv(hidden, hidden, normalize=True)
        self.linear = nn.Linear(hidden, num_classes)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.encode(x, edge_index)
        x = self.linear(x)
        return F.softmax(x, dim=1)

    def encode(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x 

In [12]:
from sklearn.utils import class_weight

if gnnTrain:
    # 加载数据（使用度数伪标签）
    nodes, labels, edges, mapp, lbl, nemap = load_lanl_data(split='train', label_type='degree')
    num_classes = len(np.unique(labels))
    print(f"Number of pseudo classes: {num_classes}")
    
    # 动态创建模型
    model = GCN(in_dim=20, hidden=32, num_classes=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    
    # 加载训练数据
    nodes, labels, edges, mapp, lbl, nemap = load_lanl_data(split='train')
    # 转换为 PyG Data 对象
    graph = Data(x=torch.tensor(nodes, dtype=torch.float).to(device),
                 y=torch.tensor(labels, dtype=torch.long).to(device),
                 edge_index=torch.tensor(edges, dtype=torch.long).to(device))
    
    # 类别权重
    class_weights = class_weight.compute_class_weight(class_weight='balanced',
                                                      classes=np.unique(labels),
                                                      y=labels)
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, reduction='mean')
    
    loader = NeighborLoader(graph, num_neighbors=[-1, -1], batch_size=5000)
    
    for epoch in range(100):
        model.train()
        total_loss = 0
        total_nodes = 0
        total_correct = 0
        for batch in loader:
            optimizer.zero_grad()
            pred = model(batch.x, batch.edge_index)
            loss = criterion(pred, batch.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch.x.size(0)
            total_nodes += batch.x.size(0)
            total_correct += (pred.argmax(1) == batch.y).sum().item()
        avg_loss = total_loss / total_nodes
        acc = total_correct / total_nodes
        print(f"Epoch {epoch}: Loss {avg_loss:.5f}, Accuracy {acc:.5f}")
        torch.save(model.state_dict(), gnn_weights)


if create_store:
    model.eval()
    # 重新加载训练数据（或者使用已有的 mapp, lbl, nemap）
    nodes, labels, edges, mapp, lbl, nemap = load_lanl_data(split='train')
    graph = Data(x=torch.tensor(nodes, dtype=torch.float).to(device),
                 y=torch.tensor(labels, dtype=torch.long).to(device),
                 edge_index=torch.tensor(edges, dtype=torch.long).to(device))
    with torch.no_grad():
        out = model.encode(graph.x, graph.edge_index).cpu().numpy()
    gnn_map = {}
    for i, node_id in enumerate(mapp):
        gnn_map[node_id] = (out[i].tolist(), list(nemap[node_id]))
    with open(emb_store_path, 'w') as f:
        json.dump(gnn_map, f)

Degree pseudo-label distribution: [3202 3234 3246]
Number of pseudo classes: 3
Epoch 0: Loss 1.09697, Accuracy 0.36390
Epoch 1: Loss 1.07700, Accuracy 0.65515
Epoch 2: Loss 1.04598, Accuracy 0.73923
Epoch 3: Loss 1.00566, Accuracy 0.78808
Epoch 4: Loss 0.96383, Accuracy 0.82489
Epoch 5: Loss 0.92432, Accuracy 0.86577
Epoch 6: Loss 0.88513, Accuracy 0.90680
Epoch 7: Loss 0.84792, Accuracy 0.93646
Epoch 8: Loss 0.81394, Accuracy 0.94816
Epoch 9: Loss 0.78289, Accuracy 0.95782
Epoch 10: Loss 0.75380, Accuracy 0.96190
Epoch 11: Loss 0.72656, Accuracy 0.97081
Epoch 12: Loss 0.70266, Accuracy 0.97442
Epoch 13: Loss 0.68211, Accuracy 0.97735
Epoch 14: Loss 0.66663, Accuracy 0.97809
Epoch 15: Loss 0.65214, Accuracy 0.98116
Epoch 16: Loss 0.63999, Accuracy 0.98340
Epoch 17: Loss 0.63163, Accuracy 0.98360
Epoch 18: Loss 0.62398, Accuracy 0.98258
Epoch 19: Loss 0.61688, Accuracy 0.98381
Epoch 20: Loss 0.61074, Accuracy 0.98592
Epoch 21: Loss 0.60619, Accuracy 0.98592
Epoch 22: Loss 0.60086, Accur

In [13]:
with open(emb_store_path, "r") as file:
    gnn_map = json.load(file)

In [ ]:
import numpy as np

def load_features_lanl(split='train', similarity_threshold=1):
    with open(emb_store_path, 'r') as f:
        gnn_map = json.load(f)
    nodes, labels, edges, mapp, lbl, nemap = load_lanl_data(split=split, label_type='malicious')
    X = []
    y = labels
    for i, node_id in enumerate(mapp):
        orig_feat = np.asarray(nodes[i]).flatten()
        if orig_feat.shape[0] != 20:
            print(f"Warning: orig_feat shape {orig_feat.shape} for node {node_id}, reset to zeros")
            orig_feat = np.zeros(20)

        if node_id in gnn_map:
            emb, stored_neis = gnn_map[node_id]
            gnn_feat = np.asarray(emb).flatten()
            # 期望 GNN 嵌入维度为 32
            if gnn_feat.shape[0] != 32:
                print(f"Warning: gnn_feat shape {gnn_feat.shape} for node {node_id}, reset to zeros")
                gnn_feat = np.zeros(32)
            curr_neis = nemap[node_id]
            jaccard = len(set(curr_neis).intersection(set(stored_neis))) / max(1, len(set(curr_neis).union(set(stored_neis))))
            if jaccard < similarity_threshold:
                gnn_feat = np.zeros(32)
        else:
            gnn_feat = np.zeros(32)

        combined = np.hstack((orig_feat, gnn_feat))
        X.append(combined)

    # 检查所有 combined 长度是否一致（应为 20+32=52）
    lens = [len(x) for x in X]
    if len(set(lens)) != 1:
        print(f"Error: inconsistent feature lengths: {set(lens)}")
        # 可选：将异常长度强行修正为52
        for idx, x in enumerate(X):
            if len(x) != 52:
                print(f"Fixing index {idx}, len={len(x)}")
                X[idx] = np.zeros(52)
    return np.array(X), y, edges, mapp

if xgbTrain:
    X_train, y_train, _, _ = load_features_lanl(split='train')
    xgb_cl = xgb.XGBClassifier()
    xgb_cl.fit(X_train, y_train)
    pickle.dump(xgb_cl, open(xgboost_weights, "wb"))
    print("XGBoost training accuracy:", accuracy_score(y_train, xgb_cl.predict(X_train)))

XGBoost training accuracy: 1.0


In [15]:
def load_pkl(fname):
    with open(fname, 'rb') as f:
        obj = pickle.load(f)
    return obj

In [ ]:
def validate_lanl():
    X_test, y_test, edges, mapp = load_features_lanl(split='test')
    xgb_cl = pickle.load(open(xgboost_weights, "rb"))
    pred = xgb_cl.predict(X_test)
    proba = xgb_cl.predict_proba(X_test)
    sorted_proba = np.sort(proba, axis=1)
    conf = (sorted_proba[:, -1] - sorted_proba[:, -2]) / sorted_proba[:, -1]
    conf = (conf - conf.min()) / conf.max()
    flag = (pred != y_test)
    scores = conf[flag].tolist()
    return scores

In [31]:
from itertools import compress
from torch_geometric import utils

def Get_Adjacent(ids, mapp, edges, hops):
    if hops == 0:
        return set()
    
    neighbors = set()
    for edge in zip(edges[0], edges[1]):
        if any(mapp[node] in ids for node in edge):
            neighbors.update(mapp[node] for node in edge)

    if hops > 1:
        neighbors = neighbors.union(Get_Adjacent(neighbors, mapp, edges, hops - 1))
    
    return neighbors

def calculate_metrics(TP, FP, FN, TN):
    FPR = FP / (FP + TN) if FP + TN > 0 else 0
    TPR = TP / (TP + FN) if TP + FN > 0 else 0

    prec = TP / (TP + FP) if TP + FP > 0 else 0
    rec = TP / (TP + FN) if TP + FN > 0 else 0
    fscore = (2 * prec * rec) / (prec + rec) if prec + rec > 0 else 0

    return prec, rec, fscore, FPR, TPR

def helper(MP, all_pids, GP, edges, mapp):
    TP = MP.intersection(GP)
    FP = MP - GP
    FN = GP - MP
    TN = all_pids - (GP | MP)
    prec, rec, fscore, FPR, TPR = calculate_metrics(len(TP), len(FP), len(FN), len(TN))
    print('strict')
    print(f"Precision: {round(prec, 2)}, Recall: {round(rec, 2)}, Fscore: {round(fscore, 2)}")

    two_hop_gp = Get_Adjacent(GP, mapp, edges, 2)
    two_hop_tp = Get_Adjacent(TP, mapp, edges, 2)
    FPL = FP - two_hop_gp
    TPL = TP.union(FN.intersection(two_hop_tp))
    FN = FN - two_hop_tp

    TP, FP, FN, TN = len(TPL), len(FPL), len(FN), len(TN)

    prec, rec, fscore, FPR, TPR = calculate_metrics(TP, FP, FN, TN)
    print('neighbor')
    print(f"Precision: {round(prec, 2)}, Recall: {round(rec, 2)}, Fscore: {round(fscore, 2)}")
    
    return TPL, FPL

In [25]:
def load_features_lanl(split='train', similarity_threshold=1):
    with open(emb_store_path, 'r') as f:
        gnn_map = json.load(f)
    nodes, labels, edges, mapp, lbl, nemap = load_lanl_data(split=split, label_type='malicious')
    X = []
    y = labels
    for i, node_id in enumerate(mapp):
        orig_feat = np.asarray(nodes[i]).flatten()
        if orig_feat.shape[0] != 20:
            print(f"Warning: orig_feat shape {orig_feat.shape} for node {node_id}, reset to zeros")
            orig_feat = np.zeros(20)

        if node_id in gnn_map:
            emb, stored_neis = gnn_map[node_id]
            gnn_feat = np.asarray(emb).flatten()
            # 期望 GNN 嵌入维度为 32
            if gnn_feat.shape[0] != 32:
                print(f"Warning: gnn_feat shape {gnn_feat.shape} for node {node_id}, reset to zeros")
                gnn_feat = np.zeros(32)
            curr_neis = nemap[node_id]
            jaccard = len(set(curr_neis).intersection(set(stored_neis))) / max(1, len(set(curr_neis).union(set(stored_neis))))
            if jaccard < similarity_threshold:
                gnn_feat = np.zeros(32)
        else:
            gnn_feat = np.zeros(32)

        combined = np.hstack((orig_feat, gnn_feat))
        X.append(combined)

    # 检查所有 combined 长度是否一致（应为 20+32=52）
    lens = [len(x) for x in X]
    if len(set(lens)) != 1:
        print(f"Error: inconsistent feature lengths: {set(lens)}")
        # 可选：将异常长度强行修正为52
        for idx, x in enumerate(X):
            if len(x) != 52:
                print(f"Fixing index {idx}, len={len(x)}")
                X[idx] = np.zeros(52)
    return np.array(X), y, edges, mapp

In [ ]:

#  加载测试集
X_test, y_test, edges, mapp = load_features_lanl(split='test')
xgb_cl = pickle.load(open(xgboost_weights, "rb"))
pred = xgb_cl.predict(X_test)
proba = xgb_cl.predict_proba(X_test)

# 计算置信度（与原 Flash 一致）
sorted_proba = np.sort(proba, axis=1)
conf = (sorted_proba[:, -1] - sorted_proba[:, -2]) / sorted_proba[:, -1]
conf = (conf - conf.min()) / (conf.max() + 1e-8)

# 实现
sorted_proba = np.sort(proba, axis=1)
conf = (sorted_proba[:, -1] - sorted_proba[:, -2]) / (sorted_proba[:, -1] + 1e-8)
conf_norm = (conf - conf.min()) / (conf.max() + 1e-8)
threshold = 0.6   # 可调节
check = (pred == y_test) & (conf_norm > threshold)
alert_flag = ~check   # 需要告警的条件：预测错误 或 置信度低
alert_ids = {mapp[i] for i, flag in enumerate(alert_flag) if flag}

all_ids = set(mapp)
gt_ids = get_malicious_in_range(DATE_OF_EVIL_LANL, END_TEST_TIME)

helper(alert_ids, all_ids, gt_ids, edges, mapp)

strict
Precision: 0.02, Recall: 0.72, Fscore: 0.04
neighbor
Precision: 0.52, Recall: 0.72, Fscore: 0.6


({'C1',
  'C10',
  'C10005',
  'C1003',
  'C1006',
  'C1014',
  'C1015',
  'C102',
  'C10405',
  'C1042',
  'C1046',
  'C10577',
  'C1065',
  'C10817',
  'C1085',
  'C1089',
  'C1096',
  'C11039',
  'C11178',
  'C1119',
  'C11194',
  'C1124',
  'C1125',
  'C113',
  'C115',
  'C11727',
  'C1173',
  'C1183',
  'C1191',
  'C12116',
  'C1222',
  'C1224',
  'C12320',
  'C12448',
  'C12512',
  'C126',
  'C1268',
  'C12682',
  'C1275',
  'C1302',
  'C1319',
  'C13713',
  'C1382',
  'C1415',
  'C143',
  'C1432',
  'C1438',
  'C1461',
  'C1477',
  'C1479',
  'C148',
  'C1482',
  'C1484',
  'C1493',
  'C1500',
  'C1503',
  'C1506',
  'C15197',
  'C152',
  'C15232',
  'C1549',
  'C1555',
  'C1567',
  'C1570',
  'C1581',
  'C16088',
  'C1610',
  'C1611',
  'C1616',
  'C1626',
  'C1632',
  'C16401',
  'C16467',
  'C16563',
  'C1710',
  'C1737',
  'C17425',
  'C17600',
  'C17636',
  'C17640',
  'C17693',
  'C177',
  'C1776',
  'C1784',
  'C1797',
  'C1810',
  'C1823',
  'C1906',
  'C1936',
  'C1964'